In [1]:
from sqlalchemy import create_engine
import pandas as pd
from dotenv import load_dotenv
import os

# 환경 변수 로드
load_dotenv()
database_url = os.getenv('DATABASE_URL')
engine = create_engine(database_url)
print("db 연결 성공!")

db 연결 성공!


In [2]:
query = '''
SELECT nct_id, allocation, intervention_model, primary_purpose, masking
FROM ctgov.designs;
'''
df_designs = pd.read_sql(query, engine)
df_designs.head()

,nct_id,allocation,intervention_model,primary_purpose,masking
0,NCT00663117,RANDOMIZED,CROSSOVER,TREATMENT,QUADRUPLE
1,NCT04121767,RANDOMIZED,PARALLEL,PREVENTION,NONE
2,NCT03765229,NA,SINGLE_GROUP,TREATMENT,NONE
3,NCT05727150,NA,SINGLE_GROUP,TREATMENT,NONE
4,NCT04175236,NaN,NaN,NaN,NaN


In [3]:
df_designs.info()

<class 'pandas.DataFrame'>
RangeIndex: 570332 entries, 0 to 570331
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   nct_id              570332 non-null  str  
 1   allocation          433224 non-null  str  
 2   intervention_model  432328 non-null  str  
 3   primary_purpose     432050 non-null  str  
 4   masking             433576 non-null  str  
dtypes: str(5)
memory usage: 21.8 MB


In [4]:
df_designs.isnull().sum()

nct_id                     0
allocation            137108
intervention_model    138004
primary_purpose       138282
masking               136756
dtype: int64

- allocation: 
-- 배정방식, 실험군과 대조군을 어떻게 나눌것인가. 
-- Randomized(무작위), Non-Randomized(비무작위), N/A(단일군 연구등 대조군이 없는 경우)
- intervention_model:
-- 개입 모델, 
-- SingleGroupAssignment 모든 참가자가 동일한 치료
-- Parallel Assignment 두 그룹으로 나눠 각각 다른 치료 (가장 흔함)
-- Crossover Assignment a치료를 먼저 받고 일정기간 후 b 치료 (자신이 자산의 대조군이됨)
-- Factorial Assignment 여러 치료 조합을 동시에 테스트
- primary_purpose:
-- 임상시험 주요 목적
-- Treatment 치료 (질병을 고치거나 완화하는 것이 목적)
-- Prevention 예방 (질병에 걸리지 않게 하는 것이 목적)
-- Diagnostic 진단 (병얼 더 잘 찾아내는 방법을 테스트)
-- Supportive Care 지지요법 (환자의 삶의 질을 높이는 것이 목적)
- masking:
-- 눈가림/블라인드
-- OpenLabel 모두 공개
-- SingleBlind 환자만 모름
-- DoubleBlind 환자와 연구자 모두 모름 (신뢰도가장높음)
-- QuadrupleBlind 데이터 분석하는 통계 전문자까지 모름

In [5]:
print(df_designs.shape)

(570332, 5)


In [6]:
# study_type이 interventional(중재연구) 만 필터링
load_path = r'C:\dev\clinicaltrials-study\data\processed\studies_clean.csv'
df_studies = pd.read_csv(load_path)
df_designs = df_designs.merge(df_studies[['nct_id', 'study_type']], on='nct_id', how='left')
df_designs = df_designs[df_designs['study_type']=='INTERVENTIONAL']
df_designs.drop(columns=['study_type'], inplace=True)
print(df_designs.shape)

(426365, 5)


In [7]:
df_designs.head()

,nct_id,allocation,intervention_model,primary_purpose,masking
0,NCT00663117,RANDOMIZED,CROSSOVER,TREATMENT,QUADRUPLE
1,NCT04121767,RANDOMIZED,PARALLEL,PREVENTION,NONE
2,NCT03765229,NA,SINGLE_GROUP,TREATMENT,NONE
3,NCT05727150,NA,SINGLE_GROUP,TREATMENT,NONE
5,NCT05037890,RANDOMIZED,PARALLEL,PREVENTION,TRIPLE


In [8]:
# 모든값을 소문자로 변경 (nct_id만 제외)
for col in df_designs.columns:
    if col !='nct_id' and df_designs[col].dtype == 'object':
        df_designs[col] = df_designs[col].str.lower()
df_designs.head()

,nct_id,allocation,intervention_model,primary_purpose,masking
0,NCT00663117,RANDOMIZED,CROSSOVER,TREATMENT,QUADRUPLE
1,NCT04121767,RANDOMIZED,PARALLEL,PREVENTION,NONE
2,NCT03765229,NA,SINGLE_GROUP,TREATMENT,NONE
3,NCT05727150,NA,SINGLE_GROUP,TREATMENT,NONE
5,NCT05037890,RANDOMIZED,PARALLEL,PREVENTION,TRIPLE


In [9]:
for col in df_designs.columns:
    if col != 'nct_id':
        # 일단 문자열로 강제 변환 후 소문자 적용
        df_designs[col] = df_designs[col].astype(str).str.lower()

df_designs.head()

,nct_id,allocation,intervention_model,primary_purpose,masking
0,NCT00663117,randomized,crossover,treatment,quadruple
1,NCT04121767,randomized,parallel,prevention,none
2,NCT03765229,na,single_group,treatment,none
3,NCT05727150,na,single_group,treatment,none
5,NCT05037890,randomized,parallel,prevention,triple


In [10]:
print(df_designs.shape)

(426365, 5)


In [11]:
# 논리적 오류 제거하기 (single group + randomized) - 데이터 무결성 검사 차원
invalid_rows = (df_designs['intervention_model'] == 'single_group') & (df_designs['allocation']=='randomized')
df_designs = df_designs.drop(df_designs[invalid_rows].index)
print(df_designs.shape)

(418689, 5)


In [12]:
df_designs.isnull().sum()

nct_id                   0
allocation            2988
intervention_model    3311
primary_purpose       5555
masking               2859
dtype: int64

In [13]:
df_designs['allocation'].unique()

<StringArray>
['randomized', 'na', 'non_randomized', nan]
Length: 4, dtype: str

In [14]:
df_designs['intervention_model'].unique()

<StringArray>
['crossover', 'parallel', 'single_group', nan, 'sequential', 'factorial']
Length: 6, dtype: str

- parallel: 병행배정. 가장 일반적. 두개 이상 그룹으로 나눈 뒤 각자 정해진 치료만 받음
- single group: 단일군 배정. 모든 참가자가 동일한 치료를 받음. 대조군 없는 초기 임상이나 희귀질환 연구에서 자주 사용. 무작위 배정이 필요 없는 특징.
- crossover: 교차 배정. 두가지 치료를 순차적으로 모두 경험
- sequential: 순차적 배정, 참가자들을 한꺼번에 모집하지 않고, 앞선 그룹의 결과에 따라 다음 그룹의 투여량, 방식 결정하며 진행하는 유연한 설계 방식.
- factorial: 요인 설계. 두가지 이상의 치료법 조합을 테스트. 치료법 간의 상호작용을 확인하고 싶을때

In [15]:
df_designs['primary_purpose'].unique()

<StringArray>
[               'treatment',               'prevention',
 'health_services_research',            'basic_science',
               'diagnostic',                        nan,
          'supportive_care',                    'other',
       'device_feasibility',                'screening',
                      'ect']
Length: 11, dtype: str

In [16]:
df_designs['primary_purpose'].value_counts()

primary_purpose
treatment                   265244
prevention                   44636
other                        24542
supportive_care              23679
basic_science                20234
diagnostic                   18154
health_services_research     11148
screening                     3892
device_feasibility            1481
ect                            124
Name: count, dtype: int64

- treatment: 치료
- prevention: 예방
- heath_services_research: 보건 서비스 연구 
- basic_science: 기초과학
- diagnostiics: 진단
- supportive_care: 지지요법(삶의질)
- device_fesibility: 장비 타당성 검토
- screening: 스크리닝
- ect: 전기경련요법

In [17]:
df_designs['masking'].unique()

<StringArray>
['quadruple', 'none', 'triple', 'single', 'double', nan]
Length: 6, dtype: str

In [18]:
# 비교대상이 없는 단일군 연구single_group의 allocation 배정방식은 무작위 non_randomized 배정을 할 필요가 없음
# allocation 값이 비어있거나 (isna), 문자열('na')로 되어 있는 경우가 많아 
# 데이터 손실을 막기위해 널값을 non_randomized값으로 대체
df_designs.loc[(df_designs['intervention_model'] == 'single_group') & (df_designs['allocation'].isna()), 'allocation'] = 'non_randomized'
df_designs.loc[(df_designs['intervention_model'] == 'single_group') & (df_designs['allocation'] == 'na'), 'allocation'] = 'non_randomized'
# 널값 nan, 'na', 'none' 모두 unknown으로 통일
df_designs['allocation'] = df_designs['allocation'].replace({'na': 'unknown'}).fillna('unknown')
df_designs['masking'] = df_designs['masking'].replace({'none': 'unknown'}).fillna('unknown')
df_designs['intervention_model'] = df_designs['intervention_model'].fillna('unknown')
df_designs['primary_purpose'] = df_designs['primary_purpose'].fillna('unknown')

In [19]:
df_designs.isnull().sum()

nct_id                0
allocation            0
intervention_model    0
primary_purpose       0
masking               0
dtype: int64

In [20]:
# 카테고리 통합
df_designs['primary_purpose'] = df_designs['primary_purpose'].replace({
    'basic_science':'research',
    'health_services_research' : 'research',
    'device_feasibility':'research',
    'screening': 'diagnostic',
    'ect':'other'
}).fillna('other')


In [21]:
print(df_designs['primary_purpose'].value_counts())

primary_purpose
treatment          265244
prevention          44636
research            32863
other               24666
supportive_care     23679
diagnostic          22046
unknown              5555
Name: count, dtype: int64


In [22]:
# 피쳐별 원핫인코딩 (파생변수 생성)
df_designs = pd.get_dummies(df_designs, columns=['allocation', 'intervention_model', 'primary_purpose', 'masking'],
                            prefix=['allocation', 'model', 'purpose', 'masking'], dtype=int)
df_designs = df_designs.groupby('nct_id').max().reset_index()
df_designs.head()

,nct_id,allocation_non_randomized,allocation_randomized,allocation_unknown,model_crossover,model_factorial,model_parallel,model_sequential,model_single_group,model_unknown,...,purpose_prevention,purpose_research,purpose_supportive_care,purpose_treatment,purpose_unknown,masking_double,masking_quadruple,masking_single,masking_triple,masking_unknown
0,NCT00000113,0,1,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,1,0
1,NCT00000114,0,1,0,0,1,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
2,NCT00000115,0,1,0,1,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
3,NCT00000116,0,1,0,0,0,1,0,0,0,...,0,0,0,1,0,0,1,0,0,0
4,NCT00000117,0,1,0,0,0,0,0,0,1,...,0,0,0,1,0,1,0,0,0,0


In [23]:
df_designs.info()

<class 'pandas.DataFrame'>
RangeIndex: 418689 entries, 0 to 418688
Data columns (total 22 columns):
 #   Column                     Non-Null Count   Dtype
---  ------                     --------------   -----
 0   nct_id                     418689 non-null  str  
 1   allocation_non_randomized  418689 non-null  int64
 2   allocation_randomized      418689 non-null  int64
 3   allocation_unknown         418689 non-null  int64
 4   model_crossover            418689 non-null  int64
 5   model_factorial            418689 non-null  int64
 6   model_parallel             418689 non-null  int64
 7   model_sequential           418689 non-null  int64
 8   model_single_group         418689 non-null  int64
 9   model_unknown              418689 non-null  int64
 10  purpose_diagnostic         418689 non-null  int64
 11  purpose_other              418689 non-null  int64
 12  purpose_prevention         418689 non-null  int64
 13  purpose_research           418689 non-null  int64
 14  purpose_support

In [24]:
# 최종 데이터셋 저장
target_path = r'C:\dev\clinicaltrials-study\data\processed'
file_name = 'designs_clean.csv'
full_save_path = os.path.join(target_path, file_name)
df_designs.to_csv(full_save_path, index=False)